# Differential Privacy and Poisoning Attacks

This notebook evaluates DP-FedAvg under clipping and Gaussian noise sweeps, assesses label-flip and garbage update poisoning attacks on site 0, and tests coordinate-wise median aggregation as a Byzantine defense.

In [1]:
import pandas as pd
import numpy as np
import torch
from src.data import load_all_sites, load_pooled_data
from src.fl import run_federated
from src.privacy import run_privacy_fl

CONFIG = {
    "exp": "e6",
    "dataset": "fed_heart_disease",
    "seeds": list(range(42, 52)),
    "rounds": 50,
    "local_epochs": 1,
    "lr": 0.001,
    "batch_size": 4,
    "clip_norm": 0.1,
    "dp_noise_multipliers": [0.0, 0.01, 0.05, 0.1, 0.3],
    "labelflip_rates": [0.2, 0.5, 1.0],
    "attack_site": 0
}

Load per-site partitioned datasets and pooled test evaluation set.

In [2]:
data = load_all_sites()
pooled_test = load_pooled_data(train=False)

Verify parity check: with zero noise multiplier and a huge clipping bound, DP-FedAvg reproduces Stage 2 plain FedAvg exactly.

In [3]:
parity_fedavg = run_federated(data, pooled_test=pooled_test, seeds=[42], rounds=5, method="fedavg")
parity_dp = run_privacy_fl(data, pooled_test=pooled_test, seeds=[42], rounds=5, clip_norm=10000.0, noise_multiplier=0.0)

df_p_fed = pd.DataFrame(parity_fedavg)
df_p_dp = pd.DataFrame(parity_dp)
parity_diff = (df_p_fed["metric_value"] - df_p_dp["metric_value"]).abs().max()
print(f"Parity check absolute difference: {parity_diff:.10f}")
assert parity_diff == 0.0
print("Parity check PASSED.")

Parity check absolute difference: 0.0000000000
Parity check PASSED.


Run DP-FedAvg noise multiplier sweep over 10 seeds.

In [4]:
all_records = []
for z in CONFIG["dp_noise_multipliers"]:
    scen = f"dp_z{z}"
    recs = run_privacy_fl(
        data=data, pooled_test=pooled_test, seeds=CONFIG["seeds"],
        rounds=CONFIG["rounds"], local_epochs=CONFIG["local_epochs"],
        lr=CONFIG["lr"], batch_size=CONFIG["batch_size"],
        clip_norm=CONFIG["clip_norm"], noise_multiplier=z,
        method="dp_fedavg", scenario=scen, exp=CONFIG["exp"]
    )
    all_records.extend(recs)
    df_temp = pd.DataFrame(recs)
    sub = df_temp[(df_temp["round"] == CONFIG["rounds"]) & (df_temp["site"] == "all")]
    print(f"{scen}: test acc = {sub['metric_value'].mean():.4f} +/- {sub['metric_value'].std():.4f}")

dp_z0.0: test acc = 0.7823 +/- 0.0103
dp_z0.01: test acc = 0.7827 +/- 0.0094
dp_z0.05: test acc = 0.7827 +/- 0.0106
dp_z0.1: test acc = 0.7850 +/- 0.0053
dp_z0.3: test acc = 0.7850 +/- 0.0081


Evaluate label-flip poisoning attacks and coordinate-wise median aggregation defense.

In [5]:
for q in CONFIG["labelflip_rates"]:
    scen = f"labelflip_q{q}"
    recs = run_privacy_fl(
        data=data, pooled_test=pooled_test, seeds=CONFIG["seeds"],
        rounds=CONFIG["rounds"], local_epochs=CONFIG["local_epochs"],
        lr=CONFIG["lr"], batch_size=CONFIG["batch_size"],
        clip_norm=None, noise_multiplier=0.0,
        attack_type="labelflip", attack_site=CONFIG["attack_site"],
        flip_q=q, use_median=False,
        method="fedavg_poison", scenario=scen, exp=CONFIG["exp"]
    )
    all_records.extend(recs)
    df_temp = pd.DataFrame(recs)
    sub = df_temp[(df_temp["round"] == CONFIG["rounds"]) & (df_temp["site"] == "all")]
    print(f"{scen} (FedAvg): test acc = {sub['metric_value'].mean():.4f} +/- {sub['metric_value'].std():.4f}")

recs_lf_med = run_privacy_fl(
    data=data, pooled_test=pooled_test, seeds=CONFIG["seeds"],
    rounds=CONFIG["rounds"], local_epochs=CONFIG["local_epochs"],
    lr=CONFIG["lr"], batch_size=CONFIG["batch_size"],
    clip_norm=None, noise_multiplier=0.0,
    attack_type="labelflip", attack_site=CONFIG["attack_site"],
    flip_q=1.0, use_median=True,
    method="median_defense", scenario="labelflip_median", exp=CONFIG["exp"]
)
all_records.extend(recs_lf_med)
df_lf_med = pd.DataFrame(recs_lf_med)
sub_lf_med = df_lf_med[(df_lf_med["round"] == CONFIG["rounds"]) & (df_lf_med["site"] == "all")]
print(f"labelflip_median (Median): test acc = {sub_lf_med['metric_value'].mean():.4f} +/- {sub_lf_med['metric_value'].std():.4f}")

labelflip_q0.2 (FedAvg): test acc = 0.7689 +/- 0.0139
labelflip_q0.5 (FedAvg): test acc = 0.7315 +/- 0.0284
labelflip_q1.0 (FedAvg): test acc = 0.3713 +/- 0.0268
labelflip_median (Median): test acc = 0.5291 +/- 0.0696


Evaluate garbage update attack under standard FedAvg and coordinate-wise median defense.

In [6]:
recs_garb_avg = run_privacy_fl(
    data=data, pooled_test=pooled_test, seeds=CONFIG["seeds"],
    rounds=CONFIG["rounds"], local_epochs=CONFIG["local_epochs"],
    lr=CONFIG["lr"], batch_size=CONFIG["batch_size"],
    clip_norm=None, noise_multiplier=0.0,
    attack_type="garbage", attack_site=CONFIG["attack_site"],
    garbage_scale=1.0, use_median=False,
    method="fedavg_poison", scenario="garbage_fedavg", exp=CONFIG["exp"]
)
all_records.extend(recs_garb_avg)
df_garb_avg = pd.DataFrame(recs_garb_avg)
sub_garb_avg = df_garb_avg[(df_garb_avg["round"] == CONFIG["rounds"]) & (df_garb_avg["site"] == "all")]
print(f"garbage_fedavg (FedAvg): test acc = {sub_garb_avg['metric_value'].mean():.4f} +/- {sub_garb_avg['metric_value'].std():.4f}")

recs_garb_med = run_privacy_fl(
    data=data, pooled_test=pooled_test, seeds=CONFIG["seeds"],
    rounds=CONFIG["rounds"], local_epochs=CONFIG["local_epochs"],
    lr=CONFIG["lr"], batch_size=CONFIG["batch_size"],
    clip_norm=None, noise_multiplier=0.0,
    attack_type="garbage", attack_site=CONFIG["attack_site"],
    garbage_scale=1.0, use_median=True,
    method="median_defense", scenario="garbage_median", exp=CONFIG["exp"]
)
all_records.extend(recs_garb_med)
df_garb_med = pd.DataFrame(recs_garb_med)
sub_garb_med = df_garb_med[(df_garb_med["round"] == CONFIG["rounds"]) & (df_garb_med["site"] == "all")]
print(f"garbage_median (Median): test acc = {sub_garb_med['metric_value'].mean():.4f} +/- {sub_garb_med['metric_value'].std():.4f}")

garbage_fedavg (FedAvg): test acc = 0.5437 +/- 0.1321
garbage_median (Median): test acc = 0.7441 +/- 0.0551


Save all results to results/e6.csv.

In [7]:
df_e6 = pd.DataFrame(all_records)
df_e6.to_csv("results/e6.csv", index=False)
print(f"Saved {len(df_e6)} rows to results/e6.csv")
final_summary = df_e6[(df_e6["round"] == CONFIG["rounds"]) & (df_e6["site"] == "all")].groupby("scenario")["metric_value"].agg(["mean", "std"])
print(final_summary)

Saved 27500 rows to results/e6.csv
